In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Okhla_Phase-2_Delhi_DPCC_2023.xlsx")

In [4]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,277.0,161.0,188.0,83.0,79.0,96.0,53.0,86.0,142.0,140.0,360.0,412.0
1,2,371.0,186.0,213.0,101.0,61.0,112.0,54.0,85.0,138.0,129.0,405.0,368.0
2,3,392.0,171.0,152.0,210.0,99.0,122.0,113.0,80.0,137.0,136.0,499.0,333.0
3,4,363.0,205.0,126.0,114.0,82.0,153.0,148.0,88.0,140.0,159.0,435.0,319.0
4,5,357.0,205.0,124.0,138.0,171.0,144.0,97.0,88.0,107.0,163.0,476.0,298.0
5,6,451.0,259.0,132.0,147.0,225.0,118.0,70.0,109.0,101.0,184.0,432.0,283.0
6,7,402.0,279.0,154.0,144.0,NaN,252.0,62.0,97.0,92.0,195.0,413.0,306.0
7,8,405.0,126.0,224.0,170.0,NaN,151.0,62.0,106.0,75.0,175.0,438.0,300.0
8,9,482.0,197.0,98.0,203.0,191.0,153.0,54.0,116.0,47.0,153.0,448.0,NaN
9,10,441.0,168.0,190.0,192.0,172.0,127.0,NaN,136.0,46.0,NaN,281.0,318.0


In [5]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   32 non-null     float64
 3   March      35 non-null     float64
 4   April      35 non-null     float64
 5   May        33 non-null     float64
 6   June       31 non-null     float64
 7   July       23 non-null     float64
 8   August     31 non-null     float64
 9   September  34 non-null     float64
 10  October    34 non-null     float64
 11  November   34 non-null     float64
 12  December   32 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [6]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [7]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [8]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [9]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,277.0,161.0,188.0,83.0,79.0,96.0,53.000000,86.0,142.0,140.0,360.0,412.0
1,2,371.0,186.0,213.0,101.0,61.0,112.0,54.000000,85.0,138.0,129.0,405.0,368.0
2,3,392.0,171.0,152.0,210.0,99.0,122.0,61.782609,80.0,137.0,136.0,499.0,333.0
3,4,363.0,205.0,126.0,114.0,82.0,153.0,61.782609,88.0,140.0,159.0,435.0,319.0
4,5,357.0,205.0,124.0,138.0,171.0,144.0,61.782609,88.0,107.0,163.0,476.0,298.0
